# Verification of the Lin–Doering bursty autoregulation model

This notebook verifies the curated BNGL model at two levels. First, it reruns BioNetGen and compares the deterministic trajectory with an independently implemented SciPy ODE system. Second, it compares a seeded BioNetGen SSA trajectory with the individual-based stationary protein distribution plotted as black triangles in Fig. 3b of Lin and Doering (2016).

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import tempfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

MODEL_ID = "bursty_autoregulated_gene_expression_lin2016"
PRIMARY = f"{MODEL_ID}.bngl"

def locate_model_dir():
    candidates = [Path.cwd(), Path.cwd() / MODEL_ID, Path.cwd() / "models" / MODEL_ID]
    for candidate in candidates:
        if (candidate / PRIMARY).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {PRIMARY} from {Path.cwd()}")

MODEL_DIR = locate_model_dir()
REFERENCE_DIR = MODEL_DIR / "reference"
print(f"Model directory: {MODEL_DIR}")

Model directory: /Users/l119605/Code/bngsim-paper/tmp/curation/bursty_autoregulated_gene_expression_lin2016


## 1. Fresh BioNetGen execution

The canonical BNGL file is copied to a temporary directory and executed there so this check does not rely on committed output files. The active action block first relaxes the ODE model for ten cell cycles, then carries that state into a seeded 1000-cell-cycle direct SSA trajectory. The fresh arrays are also checked against the curated reference outputs.

In [2]:
def find_bng2():
    candidates = [
        os.environ.get("BNG2_PL"),
        str(Path.home() / "Simulations" / "BioNetGen-2.9.3" / "BNG2.pl"),
    ]
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate)
    raise FileNotFoundError(
        "Set BNG2_PL to the BioNetGen BNG2.pl executable before running this notebook."
    )

bng2 = find_bng2()
with tempfile.TemporaryDirectory(prefix="lin2016_bng_") as tmp:
    run_dir = Path(tmp)
    run_model = run_dir / PRIMARY
    shutil.copy2(MODEL_DIR / PRIMARY, run_model)
    result = subprocess.run(
        ["perl", str(bng2), "--outdir", str(run_dir), str(run_model)],
        check=True,
        capture_output=True,
        text=True,
        timeout=120,
    )
    ode = np.loadtxt(run_dir / f"{MODEL_ID}_ode.gdat")
    ssa = np.loadtxt(run_dir / f"{MODEL_ID}_ssa.gdat")

reference_ode = np.loadtxt(REFERENCE_DIR / f"{MODEL_ID}_ode.gdat")
reference_ssa = np.loadtxt(REFERENCE_DIR / f"{MODEL_ID}_ssa.gdat")
np.testing.assert_allclose(ode, reference_ode, rtol=0.0, atol=1e-10)
np.testing.assert_allclose(ssa, reference_ssa, rtol=0.0, atol=1e-10)
print(f"BioNetGen: {result.stdout.splitlines()[0]}")
print(f"Fresh ODE samples: {len(ode):,}; fresh SSA samples: {len(ssa):,}")
print("Fresh seeded outputs match the curated references.")

BioNetGen: BioNetGen version 2.9.3
Fresh ODE samples: 361; fresh SSA samples: 20,001
Fresh seeded outputs match the curated references.


## 2. Independent deterministic implementation

For transcript copy number $m$ and protein copy number $P$, the independently coded mean-field equations are

$$\frac{dm}{dt}=r_0+r_1\frac{P^n}{K^n+P^n}-\gamma m,$$

$$\frac{dP}{dt}=\gamma Bm-\gamma_0P.$$

They are solved with SciPy's DOP853 integrator at stricter tolerances than the BioNetGen run. Errors are computed over both state variables at every reported time. The maximum relative error uses $\max(1,|y_{\mathrm{SciPy}}|)$ in the denominator so the zero initial condition is well defined.

In [3]:
r0, r1 = 2.0 / 3600.0, 10.0 / 3600.0
hill_n, K = 4.0, 200.0
gamma, gamma0 = 1.0 / 120.0, 1.0 / 3600.0
B = 40.0

def mean_field_rhs(_time, state):
    mrna, protein = state
    transcription = r0 + r1 * protein**hill_n / (K**hill_n + protein**hill_n)
    return [transcription - gamma * mrna, gamma * B * mrna - gamma0 * protein]

independent = solve_ivp(
    mean_field_rhs,
    (ode[0, 0], ode[-1, 0]),
    [0.0, 0.0],
    t_eval=ode[:, 0],
    method="DOP853",
    rtol=1e-11,
    atol=1e-13,
)
if not independent.success:
    raise RuntimeError(independent.message)

bng_states = ode[:, 1:3]
scipy_states = independent.y.T
difference = bng_states - scipy_states
max_relative_error = np.max(
    np.abs(difference) / np.maximum(1.0, np.abs(scipy_states))
)
relative_l2_error = np.linalg.norm(difference) / np.linalg.norm(scipy_states)
max_absolute_error = np.max(np.abs(difference), axis=0)

assert max_relative_error < 1e-6
assert relative_l2_error < 1e-6
print(f"Maximum relative error: {max_relative_error:.3e}")
print(f"Relative L2 error: {relative_l2_error:.3e}")
print(
    "Maximum absolute errors "
    f"(mRNA, protein): ({max_absolute_error[0]:.3e}, {max_absolute_error[1]:.3e})"
)

Maximum relative error: 5.961e-08
Relative L2 error: 4.678e-08
Maximum absolute errors (mRNA, protein): (9.855e-09, 5.600e-06)


## 3. Comparison with reported Fig. 3b data

The black-triangle individual-based series was digitized from the vector paths in Fig. 3b on PDF page 4. Axis calibration used vector coordinates $x_{\mathrm{PDF}}=75.059$ at 0 proteins, 161.809 at 400, and 248.556 at 800; vertical coordinates 660.385, 631.501, and 602.716 correspond to densities 0, 0.002, and 0.004. Triangle centers were transformed through those linear calibrations and stored in the CSV beside this notebook.

The plotted markers are about 6.5 by 5.6 PDF points, corresponding to roughly 15 proteins and $2\times10^{-4}$ density if their centers were selected visually. Vector-path centers make extraction error smaller, but binning conventions and finite simulation sampling remain. We therefore use total-variation distance after normalizing both 40-protein-bin series, with a conservative acceptance threshold of 0.15.

In [4]:
reported = np.genfromtxt(
    REFERENCE_DIR / "lin2016_fig3b_individual_based_digitized.csv",
    delimiter=",",
    names=True,
)
centers = reported["protein_count"]
reported_raw = reported["probability_density"]
bin_width = float(np.diff(centers)[0])
edges = np.r_[centers - bin_width / 2.0, centers[-1] + bin_width / 2.0]

protein_samples = ssa[:, 2]
simulated_density, _ = np.histogram(protein_samples, bins=edges, density=True)
reported_density = reported_raw / (reported_raw.sum() * bin_width)
coverage = np.mean((protein_samples >= edges[0]) & (protein_samples < edges[-1]))

tv_distance = 0.5 * np.sum(
    np.abs(simulated_density - reported_density) * bin_width
)
density_rmse = np.sqrt(np.mean((simulated_density - reported_density) ** 2))
density_mae = np.mean(np.abs(simulated_density - reported_density))

assert tv_distance <= 0.15
print(f"SSA samples within the plotted range: {coverage:.3%}")
print(f"Total-variation distance: {tv_distance:.4f} (threshold 0.15)")
print(f"Density RMSE: {density_rmse:.3e}")
print(f"Density MAE: {density_mae:.3e}")

SSA samples within the plotted range: 99.945%
Total-variation distance: 0.0463 (threshold 0.15)
Density RMSE: 1.132e-04
Density MAE: 9.264e-05


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

time_cycles = ode[:, 0] / 3600.0
ax = axes[0]
ax.plot(time_cycles, bng_states[:, 1], color="#2864a5", lw=2.0, label="Protein, BNG")
ax.plot(
    time_cycles[::18], scipy_states[::18, 1], linestyle="none", marker="o",
    markerfacecolor="none", markeredgecolor="#2864a5", markersize=4.5,
    label="Protein, SciPy",
)
ax.set_xlabel("Time (cell cycles)")
ax.set_ylabel("Protein copy number", color="#2864a5")
ax.tick_params(axis="y", colors="#2864a5")
ax.grid(alpha=0.2)

ax_mrna = ax.twinx()
ax_mrna.plot(time_cycles, bng_states[:, 0], color="#cf6b25", lw=2.0, label="mRNA, BNG")
ax_mrna.plot(
    time_cycles[::18], scipy_states[::18, 0], linestyle="none", marker="s",
    markerfacecolor="none", markeredgecolor="#cf6b25", markersize=4.0,
    label="mRNA, SciPy",
)
ax_mrna.set_ylabel("mRNA copy number", color="#cf6b25")
ax_mrna.tick_params(axis="y", colors="#cf6b25")
handles_1, labels_1 = ax.get_legend_handles_labels()
handles_2, labels_2 = ax_mrna.get_legend_handles_labels()
ax.legend(handles_1 + handles_2, labels_1 + labels_2, loc="center right", fontsize=8)
ax.set_title("Deterministic trajectory")

ax = axes[1]
ax.step(centers, simulated_density, where="mid", color="#2864a5", lw=2.0,
        label="Curated BNGL, SSA")
ax.plot(centers, reported_density, linestyle="none", marker="^", markersize=5.5,
        markerfacecolor="none", markeredgecolor="black",
        label="Lin–Doering Fig. 3b")
ax.set_xlim(0, 1000)
ax.set_ylim(bottom=0)
ax.set_xlabel("Protein copy number")
ax.set_ylabel("Probability density")
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=8)
ax.set_title(f"Stationary distribution (TV = {tv_distance:.3f})")

fig.suptitle("Bursty positive autoregulation: independent and reported-data checks")
fig.tight_layout()
figure_path = MODEL_DIR / "verify_lin2016.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved {figure_path.name}")

Saved verify_lin2016.png


## Result

Passing assertions establish that the curated BioNetGen model reproduces its committed seeded outputs, matches an independent implementation of the paper's deterministic equations to numerical tolerance, and reproduces the reported individual-based stationary distribution within the stated digitization-aware threshold.